In [3]:
import torch
from torch import nn
from torchvision.transforms import v2

In [7]:
class convolutionalBlock(nn.Module):
    def __init__(self, input_layers, output_layers, kernel_size=3, stride=1, padding=0):
      super().__init__()

      self.cnv1 = nn.Conv2d(input_layers, output_layers, kernel_size, stride=stride, padding=padding)
      self.bn1 = nn.BatchNorm2d(output_layers)

      self.cnv2 = nn.Conv2d(output_layers, output_layers, kernel_size, stride=stride, padding=padding)
      self.bn2 = nn.BatchNorm2d(output_layers)

      self.relu = nn.ReLU()

    def forward(self, x):
      x = self.cnv1(x)
      x = self.bn1(x)
      x = self.relu(x)

      x = self.cnv2(x)
      x = self.bn2(x)
      x = self.relu(x)

      return x

In [6]:
class downSampleBlock(nn.Module):
  def __init__(self, in_channels, out_channels):
    super().__init__()

    self.cnv_block = convolutionalBlock(in_channels, out_channels)
    self.maxpool = nn.MaxPool2d((2, 2), stride=2)


  def forward(self, x):
    skip = self.cnv_block(x)
    x = self.maxpool(skip)

    return x, skip

In [4]:
class upSampleBlock(nn.Module):
  def __init__(self, in_channels, out_channels):
    super().__init__()

    self.cnv_block = convolutionalBlock(in_channels, out_channels)
    self.upcnv = nn.ConvTranspose2d(
            out_channels,
            out_channels//2,
            kernel_size=2,
            stride=2
        )

  def forward(self, x, skip):
    if skip is not None:
      skip = v2.functional.center_crop(skip, x.shape[-2:])
      x = torch.cat((x, skip), dim=1)

    x_1 = self.cnv_block(x)
    x = self.upcnv(x_1)

    return x, x_1

In [5]:
class UNet(nn.Module):
  def __init__(self, class_number, in_layers=3):
    super().__init__()

    self.ccb1 = downSampleBlock(in_layers, 64)
    self.ccb2 = downSampleBlock(64, 128)
    self.ccb3 = downSampleBlock(128, 256)
    self.ccb4 = downSampleBlock(256, 512)

    self.ecb0 = upSampleBlock(512, 1024)
    self.ecb1 = upSampleBlock(1024, 512)
    self.ecb2 = upSampleBlock(512, 256)
    self.ecb3 = upSampleBlock(256, 128)
    self.ecb4 = upSampleBlock(128, 64)

    self.final_cnv = nn.Conv2d(64, class_number, 1)

  def forward(self, x):
    x, x_1 = self.ccb1(x)
    x, x_2 = self.ccb2(x)
    x, x_3 = self.ccb3(x)
    x, x_4 = self.ccb4(x)

    x, _ = self.ecb0(x, None)
    x, _ = self.ecb1(x, x_4)
    x, _ = self.ecb2(x, x_3)
    x, _ = self.ecb3(x, x_2)
    _, x = self.ecb4(x, x_1)

    x = self.final_cnv(x)

    return x

In [8]:
x = torch.rand(3, 3, 572, 572)
y = torch.rand(3, 3, 572, 572)

In [9]:
inp = torch.rand((1, 3, 572, 572))


In [10]:
unet = UNet(12)
